In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import root_mean_squared_log_error
import os
from sklearn.model_selection import KFold

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/metadata.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv


# Paths and Cross Validation

In [2]:

import os
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error

from catboost import CatBoostRegressor
from xgboost import XGBRegressor


SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)


TRAIN_PATH = "/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv"
SAMPLE_PATH = "/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv"

TARGET = "TargetValue"
ID_COL = "TransactionID"



N_SPLITS = 5

kf = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

print("=" * 60)
print("Playground Series S5E9 - CatBoost + XGBoost")
print("=" * 60)
print(f"Seed : {SEED}")
print(f"Folds: {N_SPLITS}")

Playground Series S5E9 - CatBoost + XGBoost
Seed : 42
Folds: 5


# Load Data

In [3]:

train = pd.read_csv(TRAIN_PATH,low_memory=False)
test = pd.read_csv(TEST_PATH,low_memory=False)
sample_submission = pd.read_csv(SAMPLE_PATH)

print("=" * 60)
print("DATASET SHAPES")
print("=" * 60)

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")


train_ids = train[ID_COL].copy()
test_ids = test[ID_COL].copy()


y = np.log1p(train[TARGET])


n_train = train.shape[0]
n_test = test.shape[0]


train = train.drop(columns=[TARGET])


full = pd.concat(
    [train, test],
    axis=0,
    ignore_index=True
)

print("=" * 60)
print("Combined Dataset")
print("=" * 60)
print(full.shape)

print("\nTarget transformed using log1p.")

DATASET SHAPES
Train : (138701, 50)
Test  : (15000, 49)
Combined Dataset
(153701, 49)

Target transformed using log1p.


# Advanced Feature Engineering

In [4]:



full["TransactionDate"] = pd.to_datetime(full["TransactionDate"])

full["TransactionYear"] = full["TransactionDate"].dt.year
full["TransactionMonth"] = full["TransactionDate"].dt.month
full["TransactionQuarter"] = full["TransactionDate"].dt.quarter
full["TransactionWeek"] = full["TransactionDate"].dt.isocalendar().week.astype(int)
full["TransactionDay"] = full["TransactionDate"].dt.day
full["TransactionDayOfWeek"] = full["TransactionDate"].dt.dayofweek
full["TransactionDayOfYear"] = full["TransactionDate"].dt.dayofyear

full["IsWeekend"] = (
    full["TransactionDayOfWeek"] >= 5
).astype(int)



full["AssetAge"] = (
    full["TransactionYear"] -
    full["ManufactureYear"]
)

full["AssetAge"] = full["AssetAge"].clip(lower=0)

full["LogAssetAge"] = np.log1p(full["AssetAge"])

full["AssetAgeSquared"] = (
    full["AssetAge"] ** 2
)



full["DescriptorLength"] = (
    full["Spec_FullDescriptor"]
    .fillna("")
    .str.len()
)

full["BaseClassLength"] = (
    full["Spec_BaseClass"]
    .fillna("")
    .str.len()
)

full["SubClassLength"] = (
    full["Spec_SubClass"]
    .fillna("")
    .str.len()
)

full["VariantLength"] = (
    full["Spec_VariantModifier"]
    .fillna("")
    .str.len()
)



full["HasOperationalHours"] = (
    full["OperationalHoursMeter"]
    .notna()
    .astype(int)
)

full["HasVariantModifier"] = (
    full["Spec_VariantModifier"]
    .notna()
    .astype(int)
)

full["HasDescriptor"] = (
    full["Spec_FullDescriptor"]
    .notna()
    .astype(int)
)

full["HasCabinType"] = (
    full["CabinType"]
    .notna()
    .astype(int)
)


full["OperationalHoursMeter"] = (
    full["OperationalHoursMeter"]
    .fillna(0)
)

full["LogHours"] = np.log1p(
    full["OperationalHoursMeter"]
)

full["HoursPerYear"] = (
    full["OperationalHoursMeter"] /
    (full["AssetAge"] + 1)
)

full["WearIndex"] = (
    full["AssetAge"] *
    full["LogHours"]
)


reference_date = full["TransactionDate"].min()

full["DaysSinceStart"] = (
    full["TransactionDate"] -
    reference_date
).dt.days


full["Age_x_Hours"] = (
    full["AssetAge"] *
    full["OperationalHoursMeter"]
)

full["Age_x_Days"] = (
    full["AssetAge"] *
    full["DaysSinceStart"]
)



full.drop(columns=["TransactionDate"], inplace=True)

print("=" * 60)
print("Feature Engineering Completed")
print("Current Shape :", full.shape)
print("=" * 60)

Feature Engineering Completed
Current Shape : (153701, 73)


# Missing values imputation

In [5]:



num_cols = full.select_dtypes(include=["int64", "float64"]).columns


cat_cols = full.select_dtypes(include=["object"]).columns



for col in num_cols:

    if full[col].isnull().sum() > 0:

        full[col] = full[col].fillna(
            full[col].median()
        )



for col in cat_cols:

    if full[col].isnull().sum() > 0:

        full[col] = full[col].fillna("Missing")

print("=" * 60)
print("Missing Values Remaining")
print("=" * 60)

print(full.isnull().sum().sum())

Missing Values Remaining
0


# Label Encoding

In [6]:


cat_cols = full.select_dtypes(include=["object"]).columns.tolist()

label_encoders = {}

for col in cat_cols:

    le = LabelEncoder()

    full[col] = le.fit_transform(full[col].astype(str))

    label_encoders[col] = le

print("=" * 60)
print("Label Encoding Completed")
print("=" * 60)
print(f"Encoded {len(cat_cols)} categorical columns.")

Label Encoding Completed
Encoded 43 categorical columns.


In [7]:
X = full.iloc[:len(train)]
X_test = full.iloc[len(train):]

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)


# XGBOOST - KFOLD TRAINING


In [8]:
xgb_params = {
    "n_estimators": 5000,
    "learning_rate": 0.03,
    "max_depth": 8,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "reg:squarederror",
    "random_state": SEED,

    
    "tree_method": "hist"
}
oof_xgb = np.zeros(len(X))
test_xgb = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\nXGB Fold {fold + 1}")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = XGBRegressor(**xgb_params)

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    oof_xgb[val_idx] = model.predict(X_val)
    test_xgb += model.predict(X_test) / N_SPLITS


XGB Fold 1

XGB Fold 2

XGB Fold 3

XGB Fold 4

XGB Fold 5


# CATBOOST - KFOLD TRAINING

In [9]:
cat_params = {
    "iterations": 2000,
    "learning_rate": 0.03,
    "depth": 8,
    "loss_function": "RMSE",
    "random_seed": SEED,
    "verbose": 0,
    "task_type": "CPU"
}
oof_cat = np.zeros(len(X))
test_cat = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\nCatBoost Fold {fold + 1}")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostRegressor(**cat_params)

    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        verbose=False
    )

    oof_cat[val_idx] = model.predict(X_val)
    test_cat += model.predict(X_test) / N_SPLITS


CatBoost Fold 1

CatBoost Fold 2

CatBoost Fold 3

CatBoost Fold 4

CatBoost Fold 5


# MODEL SCORES

In [10]:


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

print("XGBoost RMSE :", rmse(y, oof_xgb))
print("CatBoost RMSE:", rmse(y, oof_cat))

XGBoost RMSE : 0.2009710396860869
CatBoost RMSE: 0.21871096820195185



# OPTIMIZE BLEND WEIGHTS (CatBoost + XGBoost)



In [11]:

best_w = 0
best_score = 1e18

print("=" * 60)
print("BLEND OPTIMIZATION")
print("=" * 60)

for w in np.arange(0, 1.01, 0.01):

    blended = w * oof_cat + (1 - w) * oof_xgb
    score = np.sqrt(mean_squared_error(y, blended))

    if score < best_score:
        best_score = score
        best_w = w

print(f"Best Weight (CatBoost): {best_w:.2f}")
print(f"Best CV RMSE: {best_score:.5f}")

BLEND OPTIMIZATION
Best Weight (CatBoost): 0.10
Best CV RMSE: 0.20075


# Final Blended Prediction

In [12]:


final_oof = best_w * oof_cat + (1 - best_w) * oof_xgb
final_test = best_w * test_cat + (1 - best_w) * test_xgb

print("Final OOF RMSE:", np.sqrt(mean_squared_error(y, final_oof)))

Final OOF RMSE: 0.20074660382657772


# APPLY BEST BLEND WEIGHT

In [13]:

print("=" * 60)
print("FINAL MODEL EVALUATION")
print("=" * 60)

final_oof = best_w * oof_cat + (1 - best_w) * oof_xgb
final_test = best_w * test_cat + (1 - best_w) * test_xgb

final_score = np.sqrt(mean_squared_error(y, final_oof))

print(f"Best Weight (CatBoost): {best_w}")
print(f"Final Blended RMSE   : {final_score}")

FINAL MODEL EVALUATION
Best Weight (CatBoost): 0.1
Final Blended RMSE   : 0.20074660382657772


# CREATE SUBMISSION FILE

In [14]:


submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: np.expm1(final_test)
})

submission.to_csv("submission.csv", index=False)

print("Submission saved successfully!")
submission.head()

Submission saved successfully!


,TransactionID,TargetValue
0,1139307,65418.446186
1,1139419,81152.556226
2,1139482,32526.254885
3,1139522,19886.584192
4,1139684,13067.659976
